In [ ]:
import gymnasium as gym
from gymnasium_env import BlokusEnv
import numpy as np
from tqdm import tqdm
import pickle
import cProfile
import time

from src.agents import Agent, RandomAgent, SimpleFunctionAgent, MiniMaxAgent, QL_Agent
from gymnasium_env.envs.single_agent_blokus_env import SingleAgentBlokusEnv
from src.agents import ABPruningAgent, MiniMaxAgent
# from src.agents.ab_pruning.ab_agent import my_heu
from src.utils import encode_board_string, decode_board_string, env_action_context
from src.agents.ab_pruning.heuristics import minimise_opponent_expanders_, maximise_my_expanders_, level_7, my_heu, mine_, mine2_

### Initialising αβ-pruning Agent :))

In [ ]:
BOARD_SIZE = 7

def mine(env, obs, action, depth):
    if depth < 2 or depth > 8:
        return level_7(env, obs, action)
    with env_action_context(env, action) as new_obs:
        return (
            mine_(obs, new_obs, action) * mine2_(obs, new_obs, action),
        )
    
abpruning_agent = ABPruningAgent(
    board_size=BOARD_SIZE, 
    depth=-1, 
    use_cache=True,
    sorted_order=lambda x:mine(*x),
    # testing_mode=(True, 5)
)
random_agent = RandomAgent()

### Initialising SingleAgent Environment :))

In [ ]:
hidden_agents = [None, None]
HIDDEN_AGENT_ID = 2 # <- 2
hidden_agents[HIDDEN_AGENT_ID - 1] = abpruning_agent

env = SingleAgentBlokusEnv(
    # hidden_agents=[None, abpruning_agent],
    hidden_agents=hidden_agents,
    # hidden_agents=[None, random_agent],
    board_size=BOARD_SIZE,
    player_turn=3-HIDDEN_AGENT_ID # <- 1
)

In [ ]:
import numpy as np
import pickle
import matplotlib.pyplot as plt

# Assuming self.log is a list of tuples (num_pruned, visited_states, pruned_percentage)

# Plot the data
plt.figure(figsize=(20, 12))
K = 1

# with open('logs3.pkl', 'rb') as f:
#     logs = pickle.load(f)
logs = env.hidden_agents[HIDDEN_AGENT_ID].log
max_length = 0
max_name = None
INF = 1e9
# min_length = [(INF) for _ in range(5)]
# min_name = ["" for _ in range(5)]
# for name, log in logs.items():
#     log, timed = log[0], log[1]
#     # print(log)
#     # print(name, timed)
#     if max_length < len(log):
#         max_length = len(log)
#         max_name = name
    # if min_length[int(name[6])] > timed:
    #     min_length[int(name[6])] = timed
    #     min_name[int(name[6])] = name
log = logs
num_pruned = np.array([entry[0] for entry in log])
visited_states = np.array([entry[1] for entry in log])
pruned_percentage = np.array([entry[2] for entry in log])
# plt.scatter(num_pruned[::K], pruned_percentage[::K], marker='.')
plt.plot(num_pruned[::K], pruned_percentage[::K], linewidth=0.5)
print(max_name)
# for i in range(1, 5):
#     print(min_length[i], min_name[i])
plt.xlabel('Visited States')
plt.legend()
plt.ylabel('Pruned Percentage')
plt.title('Pruned Percentage vs Visited States')
plt.grid(True)
plt.savefig('pruned_percentage_vs_visited_states.png')
plt.show()

In [ ]:
alpha = 0.01  # Learning rate
# min_alpha = 0.01
gamma = 0.995  # Discount factor
epsilon = 1.0  # Exploration rate
epsilon_decay = 0.9995
min_epsilon = 0.01

ql_agent = QL_Agent(
    name="ql_agent",
    q_table_path=f"against_alpha_beta_{BOARD_SIZE}.pkl",
    alpha=alpha,
    gamma=gamma,
    epsilon=epsilon,
    epsilon_decay=epsilon_decay,
    min_epsilon=min_epsilon,
    parameter_update_frequency=100,
    estimated_steps=BOARD_SIZE * 2
)

In [ ]:
a = 0
win_accuracy = 70

def main(num_episodes = 1):
    global a
    # env.render_mode = "console"

    win_counter, tie_counter, lose_counter = 0, 0, 0
    win_rate, tie_rate, loss_rate = [], [], []
    last_win_counter = 0
    # env.render_mode = "human"
    pbar = tqdm(range(num_episodes))
    try:
        states_before = len(env.hidden_agents[HIDDEN_AGENT_ID].cache_manager.cache)
    except:
        states_before = 0
    for i in pbar:
        obs, info = env.reset()
        state = encode_board_string(obs["state"])
        done = False
        # total_reward = 0
        total_reward = obs["points"][env.player_turn] - obs["points"][3 - env.player_turn]
        while not done:
            action = ql_agent.get_action(env, obs)
            obs, reward, terminated, truncated, info = env.step(action)
            next_state = encode_board_string(obs["state"])
            total_reward += reward
            done = terminated or truncated
            ql_agent.learn(state, action, reward, next_state, done, obs["steps"] - 1)
            state = next_state
            if terminated:
                assert total_reward == obs["points"][env.player_turn] - obs["points"][3 - env.player_turn], f"Total reward mismatch: {total_reward} != {obs['points'][env.player_turn]} - {obs['points'][3 - env.player_turn]}, board: {obs["state"]}"
                win_counter += total_reward > 0
                tie_counter += total_reward == 0
                lose_counter += total_reward < 0
            if truncated:
                raise Exception("Truncated")
        if (i + 1) % 100 == 0:
            last_win_counter = win_counter
            try:
                cache_size = len(env.hidden_agents[HIDDEN_AGENT_ID].cache_manager.cache)
            except:
                cache_size = 0
            pbar.set_description(f"E {i+1}: W: {win_counter}, T: {tie_counter}, L: {lose_counter}, ε: {ql_agent.epsilon:.4f}, S: {len(ql_agent.q_table)}, Cache: {cache_size}")
            win_counter = 0
            tie_counter = 0
            lose_counter = 0
        if i == num_episodes - 100 - 1:
            ql_agent.test_mode()

    global win_accuracy
    ql_agent.save_q_table()
    print(f"Q-table saved with accuracy {last_win_counter}")
    try:
        states_after = len(env.hidden_agents[HIDDEN_AGENT_ID].cache_manager.cache)
        env.save_caches()
        print(f"States discovered: {states_after - states_before}")
        # print(states_before, states_after)
    except Exception as e:
        print(e)
        pass

    # win_accuracy = last_win_counter
    # if last_win_counter > 80:
    #     print("Win rate above 95%")
    # else:
    #     # main(num_episodes)
    #     a += 1

In [ ]:
import pstats

number_of_episodes = 10000 * ql_agent.parameter_update_frequency
ql_agent.test_mode()
# env.render_mode = "human"
try:
    # Run the profiler
    cProfile.run(f"main({number_of_episodes})", "profile_output")
finally:
    # Create a Stats object
    p = pstats.Stats("profile_output")

    # Set the precision for the output
    p.strip_dirs().sort_stats("cumtime").print_stats()

    # Print the stats with custom formatting
    for func in p.fcn_list:
        stats = p.stats[func]
        ncalls = stats[0]
        tottime = stats[2]
        percall = tottime / ncalls if ncalls else 0
        cumtime = stats[3]
        filename = func[0]
        lineno = func[1]
        funcname = func[2]
        print(f"{ncalls:>8} {tottime:>8.3f} {percall:>8.3f} {cumtime:>8.3f} {percall:>8.8f} {filename}:{lineno}({funcname})")

    print(a)
    # TODO: THERE IS A BUG HERE, LOCATE BEFORE CONTINUING

In [ ]:
import numpy as np
import pickle
import matplotlib.pyplot as plt

# Assuming self.log is a list of tuples (num_pruned, visited_states, pruned_percentage)

# Plot the data
plt.figure(figsize=(20, 12))
K = 100000

# with open('logs3.pkl', 'rb') as f:
#     logs = pickle.load(f)
logs = env.hidden_agents[HIDDEN_AGENT_ID].log
print(logs)
max_length = 0
max_name = None
INF = 1e9
# min_length = [(INF) for _ in range(5)]
# min_name = ["" for _ in range(5)]
# for name, log in logs.items():
#     log, timed = log[0], log[1]
#     # print(log)
#     # print(name, timed)
#     if max_length < len(log):
#         max_length = len(log)
#         max_name = name
    # if min_length[int(name[6])] > timed:
    #     min_length[int(name[6])] = timed
    #     min_name[int(name[6])] = name
log = logs
num_pruned = np.array([entry[0] for entry in log])
visited_states = np.array([entry[1] for entry in log])
pruned_percentage = np.array([entry[2] for entry in log])
# plt.scatter(num_pruned[::K], pruned_percentage[::K], marker='.')
plt.plot(num_pruned[::K], pruned_percentage[::K], linewidth=0.5)
print(max_name)
# for i in range(1, 5):
#     print(min_length[i], min_name[i])
plt.xlabel('Visited States')
plt.legend()
plt.ylabel('Pruned Percentage')
plt.title('Pruned Percentage vs Visited States')
plt.grid(True)
plt.savefig('pruned_percentage_vs_visited_states.png')
plt.show()

In [ ]:
# display freq
# states_after = len(env.agent.cache)
import matplotlib.pyplot as plt

# print(freq)
freq = np.array(freq)
last_non_zero_index = 233
print(f"Last index different from 0: {last_non_zero_index}")
# Plot the frequency distribution
# plt.figure(figsize=(10, 6))
# l, r = 0, 234
# plt.bar(range(l, r), freq[l: r], color='blue')
# plt.xlabel('State Length')
# plt.ylabel('Frequency')
# plt.title('Frequency Distribution of State Lengths')
# plt.grid(True)
# plt.show()

def plot_cdf_with_condition(f):
    indices = [x for x in range(1000) if f(x)]
    print(indices)
    plt.figure(figsize=(10, 6))
    plt.plot(indices, freq[indices], marker='.', linestyle='none')
    plt.xlabel('Frequency')
    plt.ylabel('CDF')
    plt.title('CDF of Frequencies with Condition')
    plt.grid(True)
    plt.show()

plot_cdf_with_condition(lambda x: freq[x] > 2500)
# # print(f"States discovered: {states_after - states_before}")
# env.agent.save_cache()
# print(states_after)
# with open(AGENT_NAME, 'wb') as f:
#     pickle.dump(q_table, f)

In [ ]:
# print(env.agent.cache)

In [ ]:
import random
from src.agents.agent_heuristics import greedy

# AGENT_NAME = 'q_table6.pkl'
agent = QL_Agent(name = "ql_agent", q_table_path="against_minimax2.pkl")
# with open(AGENT_NAME, 'rb') as f:
#     q_table = pickle.load(f)
agent.epsilon = 0
random_agent1 = RandomAgent()
greedy_agent1 = SimpleFunctionAgent(name="hey", func=greedy)
helper_agent = random_agent1
# opponent_agent = gree
random_agent = RandomAgent()
greedy_agent = SimpleFunctionAgent(name="hey", func=greedy)
minimax_agent = MiniMaxAgent(name="minimax", depth=2, board_size=7)
# env.agent = greedy_agent
# env.render_mode = "human"
env.render_mode = "console"
num_states = 0
not_visited = 0
def test_agent(env, num_episodes=100):
    global q_table
    global not_visited
    win_counter, tie_counter, lose_counter = 0, 0, 0
    global num_states
    for _ in range(num_episodes):
        obs, info = env.reset()
        state = encode_board_string(obs["state"])
        done = False
        total_reward = 0

        while not done:
            num_states += 1
            action = agent.get_action(env, obs)
            if action == -1:
                action = helper_agent.get_action(env, obs)
                not_visited += 1
            next_obs, reward, terminated, truncated, info = env.step(action)
            next_state = encode_board_string(next_obs["state"])
            state = next_state
            obs = next_obs
            total_reward += reward
            done = terminated or truncated
            # env.render()
        if total_reward > 0:
            win_counter += 1
        elif total_reward == 0:
            tie_counter += 1
        else:
            lose_counter += 1

    return win_counter, tie_counter, lose_counter

# print(ql_agent.get_q_value("0" * 49, 58))
# print(ql_agent.argmax("0" * 49))

win_counter1 = np.zeros(1000)
tie_counter1 = np.zeros(1000)
lose_counter1 = np.zeros(1000)
# Test the agent
prange = tqdm(range(1000))
for i in prange:
    win_counter1[i], tie_counter1[i], lose_counter1[i] = test_agent(env)
    prange.set_description(f"W: {win_counter1[:i+1].mean():.3f}, T: {tie_counter1[:i+1].mean():.3f}, L: {lose_counter1[:i+1].mean():.3f}, NV: {not_visited / (num_states) * 100:.3f}%")

print(f"Number of not visited states: {not_visited}")

In [ ]:
import matplotlib.pyplot as plt

win_counter = win_counter1
tie_counter = tie_counter1
lose_counter = lose_counter1
non_losing_counter = win_counter + tie_counter
# Calculate the CDF
win_counter_sorted = np.sort(win_counter)
cdf = np.arange(1, len(win_counter_sorted) + 1) / len(win_counter_sorted)

# Plot the CDF for wins
plt.figure(figsize=(10, 6))
plt.plot(win_counter_sorted, cdf, marker='.', linestyle='none', label='Wins', markersize=1)
plt.xlabel('Number of Wins')
plt.ylabel('CDF')
plt.title('CDF of Win Rate for Each of the 1000 Set of 100 Games')
plt.grid(True)

num_wins_less_than_75 = np.sum(win_counter < 75)
print(f"Number of win_counter less than 75: {num_wins_less_than_75}")

# Calculate and plot the CDF for non-losing counter
non_losing_counter_sorted = np.sort(non_losing_counter)
cdf_non_losing = np.arange(1, len(non_losing_counter_sorted) + 1) / len(non_losing_counter_sorted)
plt.plot(non_losing_counter_sorted, cdf_non_losing, marker='.', linestyle='none', label='Non-losing', markersize=1)

plt.legend()
plt.show()

In [ ]:
print(sum(win_counter),sum(non_losing_counter))